# Your first RL run

In this tutorial you train Qwen/Qwen3.5-9B with reinforcement learning to multiply. The run's own sampler draws several answers to each problem, your program scores them, and the run trains on the scores, then publishes its new weights back to the sampler. You write each round yourself, with the SDK alone, and run it four times.

[Your first fine-tune](102_first_sft.ipynb) trains on answers you wrote; here the model's own answers are the data. [Operations](103_operations.ipynb) explains an operation's two awaits, which every call below uses. Put your API key in the settings cell's `API_KEY`.

## Install the SDK and tokenizer

`transformers` provides tokenization only; this notebook never loads model weights locally.

In [1]:
%pip install -q "goaly-looper==0.1.0b5" "transformers==5.19.0"

## Settings

The model, its tokenizer, where Looper is, your project and your API key. Then the run: the LoRA's rank and learning rate, how many problems a round samples, how many answers to each, how many rounds, and the longest answer.

In [2]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"
LORA_RANK = 32
LEARNING_RATE = 8e-5
PROBLEMS = 8
GROUP_SIZE = 8
ROUNDS = 4
MAX_TOKENS = 64

## Problems and their reward

The problems multiply two three-digit numbers. A prompt asks for the product under the model's chat template, with thinking off, as token ids.

The reward is 1 when the last number in an answer is the product, else 0. Your program computes it from the decoded text, and the service sees it only in the datums you send.

In [3]:
import asyncio
import random
import re

from transformers import AutoTokenizer

from looper_sdk import AdamW, Datum, Grpo, Lora, SamplingParams, ServiceClient, TrainingConfig

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
rng = random.Random(0)
problems = [(rng.randint(100, 999), rng.randint(100, 999)) for _ in range(PROBLEMS)]


def prompt(a, b):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": f"What is {a} × {b}? Answer with the number only."}],
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,
    )
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def reward(sequence, a, b):
    text = tokenizer.decode(sequence.token_ids, skip_special_tokens=True)
    numbers = re.findall(r"\d+", text.replace(",", ""))
    return float(bool(numbers) and int(numbers[-1]) == a * b)


prompts = [prompt(a, b) for a, b in problems]
longest = max(len(tokens) for tokens in prompts) + MAX_TOKENS
a, b = problems[0]
print(f"{len(problems)} problems, such as {a} × {b} = {a * b}; at most {longest} tokens each")

8 problems, such as 964 × 494 = 476216; at most 94 tokens each


## The run and its sampler

`TrainingConfig` names the model, the LoRA and its optimizer, AdamW at `LEARNING_RATE`. `tokens_per_step` holds one round's datums: every answer, each with its prompt.

A run trains. To sample its own weights it needs a sampler of its own, which its training client creates. The sampler's sizes pick its GPUs: the longest sequence, and how many answers it generates at once. Qwen3.5-9B has sizes for this, many short sequences at once. The sampler serves nothing until the run publishes.

In [4]:
config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="first-rl"
)
training = TrainingConfig(
    model=MODEL,
    kind=Lora(rank=LORA_RANK),
    tokens_per_step=PROBLEMS * GROUP_SIZE * longest,
    max_seq_len=longest,
    optimizer=AdamW(learning_rate=LEARNING_RATE, betas=(0.9, 0.95), max_grad_norm=1.0),
)
params = SamplingParams(
    max_tokens=MAX_TOKENS, temperature=1.0, stop_token_ids=(tokenizer.eos_token_id,)
)

## One round

A round is one step of RL, written out:

1. **Publish.** `publish_weights()` lands the run's latest weights on its sampler. Its result names the version the sampler now serves: the run's step.
2. **Sample.** `sample(prompt, n=GROUP_SIZE)` asks for a group of answers to one problem. Every problem's request goes in before any result is awaited, so the sampler generates them together. Each answer names the version that generated it.
3. **Score.** The reward function scores each answer.
4. **Train.** Each answer becomes a datum: the prompt as context with weight 0, and the answer's tokens with weight 1 and, in the `logprobs` column, the log probabilities they were sampled at. It carries its reward, and the answers to one problem share a `group_id`. `Grpo` compares each answer's reward with the rest of its group, on the trainer, and `optim_step` applies the step.

A group whose answers all score the same trains nothing: `Grpo` learns from the differences within a group. So each round prints how many groups varied.

In [5]:
async def one_round(training_client, sampler):
    published = await (await training_client.publish_weights()).result(timeout_s=300)

    # Submit every problem's group, then wait for all of them.
    requests = await asyncio.gather(
        *(sampler.sample(tokens, params=params, n=GROUP_SIZE) for tokens in prompts)
    )
    results = await asyncio.gather(*(request.result(timeout_s=300) for request in requests))

    data, rewards, varied = [], [], 0
    for index, ((a, b), tokens, result) in enumerate(zip(problems, prompts, results, strict=True)):
        group = [reward(sequence, a, b) for sequence in result.sequences]
        rewards += group
        varied += len(set(group)) > 1
        for sequence, score in zip(result.sequences, group, strict=True):
            answer = list(sequence.token_ids)
            data.append(
                Datum(
                    tokens=[*tokens, *answer],
                    weights=[0] * len(tokens) + [1] * len(answer),
                    columns={"logprobs": [0.0] * len(tokens) + list(sequence.logprobs)},
                    reward=score,
                    group_id=f"problem-{index}",
                    group_size=len(result.sequences),
                )
            )
    versions = sorted(
        {sequence.version.step for result in results for sequence in result.sequences}
    )
    mean = sum(rewards) / len(rewards)
    print(
        f"the sampler serves step {published.version.step}; {len(rewards)} answers from step "
        f"{versions}: mean reward {mean:.3f}, {varied} of {len(problems)} groups varied"
    )

    await training_client.forward_backward(data, Grpo())
    step = await (await training_client.optim_step()).result(timeout_s=300)
    print(f"trained step {step.step}")

## Train

The rounds run in one session. Starting the run and its sampler can take several minutes, while GPUs are allocated and the model loads. Leaving `async with` closes the session, which ends the run and its sampler, also when a call fails.

Eight problems make a noisy reward: what to watch is the loop's shape, each round sampling the step the one before it trained.

In [6]:
async def train():
    async with await config.build() as service:
        print("Starting the run and its sampler; this can take several minutes...")
        training_client = await service.create_training_client(training, timeout_s=20 * 60)
        sampler = await training_client.create_sampling_client(
            max_seq_len=longest, max_concurrent_sequences=PROBLEMS * GROUP_SIZE, timeout_s=20 * 60
        )
        for _ in range(ROUNDS):
            await one_round(training_client, sampler)


await train()

Starting the run and its sampler; this can take several minutes...
the sampler serves step 0; 64 answers from step [0]: mean reward 0.984, 1 of 8 groups varied
trained step 1
the sampler serves step 1; 64 answers from step [1]: mean reward 0.938, 1 of 8 groups varied
trained step 2
the sampler serves step 2; 64 answers from step [2]: mean reward 0.984, 1 of 8 groups varied
trained step 3
the sampler serves step 3; 64 answers from step [3]: mean reward 1.000, 0 of 8 groups varied
trained step 4


## What the rounds show

Qwen3.5-9B already multiplies three-digit numbers: nearly every answer is right, so a round has at most one group whose answers differ, and it trains little. The reward moves by noise, not by learning.

That is the lesson to take to a task of your own. `Grpo` learns only where a model sometimes succeeds and sometimes fails on the same prompt. On a task it always solves, or always fails, every group scores alike and nothing trains. Pick problems, or a difficulty, that the model gets right about half the time, then watch the number of groups that vary as well as the mean reward.

## Next steps

The run and its sampler ended with the session, and nothing was saved. [Your first fine-tune](102_first_sft.ipynb) shows how to save a run's weights and sample them later.

- [The RL loop](302_rl_loop.ipynb) runs these rounds with the cookbook's loop, `train_rl.train`, on grade-school math problems.
- [Operations](103_operations.ipynb) shows how to submit steps ahead, so the GPUs never wait for your program.